## Day 3 - Part 4: Transformer 기본 개념: 어텐션, 모든 것의 시작

### 개요

Day 3의 앞선 파트들에서 우리는 순서가 있는 데이터를 처리하는 강력한 도구인 RNN, LSTM, GRU를 배웠습니다.

이 모델들은 '기억'을 가진 신경망처럼, 이전 단계의 정보를 현재 단계로 넘겨주며 시계열 예측과 텍스트 분류 같은 문제를 성공적으로 해결했습니다. 

하지만 이들에게는 치명적인 약점이 있었습니다. 

바로 `순차적으로만 정보를 처리해야 한다는 점`과 `'장기 기억상실증' 문제`입니다. 문장이 길어질수록 맨 앞의 중요한 단어는 잊히기 쉽고, 모든 단어를 하나씩 순서대로 처리해야 하기에 GPU의 병렬 처리 능력을 제대로 활용하지 못해 학습이 느렸습니다.

"The animal didn't cross the street because `it` was too tired." 라는 문장에서 'it'이 'animal'을 가리킨다는 것을 RNN은 어떻게 기억할까요? 두 단어가 멀리 떨어져 있다면, 그 연결 고리는 희미해질 수밖에 없습니다.

2017년, 구글의 연구원들은 "Attention Is All You Need"라는 도발적인 제목의 논문을 통해 이 모든 문제를 해결할 혁신적인 아키텍처, `트랜스포머(Transforme)`를 세상에 내놓았습니다. 

<img src="https://gaussian37.github.io/assets/img/dl/concept/transformer/0.png">


트랜스포머는 RNN의 순환 구조를 과감히 버리고, '`어텐션(Attention)`'이라는 메커니즘 하나만으로 문장 내 모든 단어의 관계를 한 번에 파악합니다. 

어떤 단어가 문장 내 다른 모든 단어와 직접 '대화'하며 자신의 의미를 풍부하게 만드는 방식이죠. 

이 덕분에 병렬 처리가 가능해져 학습 속도는 비약적으로 빨라졌고, 아무리 멀리 떨어진 단어 사이의 관계도 놓치지 않게 되었습니다.

이번 파트에서는 현대 NLP의 패러다임을 바꾼 트랜스포머의 심장부로 깊숙이 들어가 봅니다. 

단어들이 서로에게 '주목'하는 원리인 `셀프 어텐션(Self-Attention)`부터, 여러 관점에서 문장을 바라보는 `멀티헤드 어텐션(Multi-Head Attention)`, 

그리고 순서 정보를 주입하는 `포지셔널 인코딩(Positional Encoding)`까지, 트랜스포머를 구성하는 핵심 부품들을 하나씩 분해하고 직접 코드로 조립해 볼 것입니다. 

최종적으로는 우리가 직접 만든 트랜스포머 인코더를 활용하여, Part 2에서 LSTM으로 풀었던 IMDB 영화 리뷰 감성 분석 문제를 다시 한번 해결해 봅니다. 

LSTM과 트랜스포머, 두 거인의 접근 방식과 성능을 직접 비교해볼 절호의 기회입니다.

`이번 파트의 학습 목표:`

  * RNN의 근본적인 한계(순차 처리, 장기 의존성)를 설명하고, 트랜스포머가 이를 어떻게 해결하는지 이해합니다.
  
  * 트랜스포머의 핵심 원리인 `셀프 어텐션(Self-Attention)`의 개념을 이해하고, `Query, Key, Value`의 역할을 설명할 수 있습니다.
  * `멀티헤드 어텐션(Multi-Head Attention)`이 왜 필요한지 이해하고, 어떻게 여러 관점의 정보를 통합하는지 설명할 수 있습니다.
  * 순서 정보가 없는 트랜스포머에 위치 정보를 부여하는 `포지셔널 인코딩(Positional Encoding)`의 필요성과 원리를 이해합니다.
  * 어텐션, 피드포워드 신경망, 잔차 연결, 층 정규화가 결합된 `트랜스포머 인코더 블록`의 전체 구조를 이해하고 코드로 구현할 수 있습니다.
  * IMDB 영화 리뷰 데이터셋을 활용하여, 직접 만든 트랜스포머 인코더 기반의 `감성 분류 모델`을 처음부터 끝까지 구축하고 학습시킬 수 있습니다

3Blue1Brown 트랜스포머의 기초 개념(한글 요약 동영상) : [영상 링크](https://www.youtube.com/watch?v=HnvitMTkXro)

3Blue1Brown 트랜스포머 기초 개념(원본) : [영상 링크](https://www.youtube.com/watch?v=eMlx5fFNoYc)


### 1. 왜 트랜스포머인가?: RNN을 넘어서

트랜스포머를 배우기 전, 왜 우리가 RNN, LSTM의 세계를 떠나 새로운 아키텍처로 넘어가야 하는지 그 이유를 명확히 해야 합니다.

  * `병렬화의 어려움`: RNN은 $t$ 시점의 계산이 끝나야 $t+1$ 시점의 계산을 시작할 수 있는 순차적 구조입니다. 이는 GPU가 수천 개의 코어를 놀리게 만드는 비효율을 낳고, 대규모 데이터 학습을 매우 느리게 만듭니다.
  
  * `장기 의존성 문제`: LSTM과 GRU가 이 문제를 많이 완화했지만, 여전히 정보는 한 단계씩 전달됩니다. 수백, 수천 개의 단어로 이루어진 긴 문서에서 맨 처음 나온 핵심 정보가 마지막까지 온전히 전달되기는 매우 어렵습니다.

트랜스포머는 이 문제들을 `어텐션`으로 정면 돌파합니다. 

입력된 문장의 모든 단어를 한 번에, 동시에 처리하면서 각 단어가 다른 모든 단어와 직접적인 연결을 맺게 합니다. 

"it"이라는 단어는 더 이상 "animal"의 정보가 여러 단계를 거쳐 희미하게 전달되기를 기다릴 필요 없이, 직접 "animal"을 '쳐다보고' 그 의미를 가져올 수 있습니다. 

이것이 바로 트랜스포머가 더 빠르고, 더 똑똑한 이유입니다.



### 2. 셀프 어텐션(Self-Attention): 단어들이 서로 대화하는 법

셀프 어텐션은 트랜스포머의 가장 핵심적인 아이디어입니다. 

문장 안의 한 단어가 자신의 의미를 더 잘 표현하기 위해, 문장 내 다른 모든 단어들을 얼마나 '주목'해야 할지 스스로 결정하는 메커니즘입니다.

이 과정은 세 가지 역할의 벡터, `Query, Key, Value`를 통해 이루어집니다.

1.  `Query (Q)`: 현재 내가(단어) 찾고 있는 정보는 무엇인가? (분석의 주체)

2.  `Key (K)`: 내가(다른 단어) 어떤 정보를 가지고 있는가? (정보의 제목, 검색 대상)
3.  `Value (V)`: 내가(다른 단어) 가진 실제 정보의 내용은 무엇인가? (정보의 내용)

각 단어는 자신의 임베딩 벡터로부터 이 세 가지 벡터(Q, K, V)를 모두 만들어냅니다. 

그리고 어텐션 계산은 다음과 같은 4단계로 진행됩니다.

1.  `Score 계산`: 특정 단어의 Query(Q)와 다른 모든 단어의 Key(K)를 내적(dot product)하여 '관련도 점수'를 계산합니다. Q와 K가 비슷할수록 점수가 높게 나옵니다.

2.  `스케일링`: 점수들이 너무 커지는 것을 방지하기 위해 Key 벡터 차원의 제곱근($\sqrt{d_k}$)으로 나누어 안정화시킵니다.
3.  `가중치(Weight) 계산`: 스케일링된 점수에 소프트맥스(Softmax) 함수를 적용하여 합이 1인 '어텐션 가중치'를 얻습니다. 이 가중치가 바로 각 단어를 얼마나 주목할지에 대한 비율입니다.
4.  `가중합(Weighted Sum)`: 각 단어의 Value(V) 벡터에 해당 어텐션 가중치를 곱하여 모두 더합니다.


<img src="https://oopy.lazyrockets.com/api/v2/notion/image?src=https%3A%2F%2Fs3-us-west-2.amazonaws.com%2Fsecure.notion-static.com%2Fd0eedca1-ddd9-4555-80d1-df8cfc0ede53%2FUntitled.png&blockId=98ba04e7-d65f-467d-8470-12cf6b4dae71">

결과적으로, 나와 관련성이 높은 단어들의 Value는 높은 가중치로 더해지고, 관련 없는 단어들의 Value는 낮은 가중치로 더해져 거의 무시됩니다. 

이렇게 만들어진 최종 벡터는 문장 전체의 문맥이 풍부하게 반영된 새로운 '나(단어)'의 표현이 됩니다.

#### 코드 실습: `Self-Attention` 모듈 구현하기

이 개념을 코드로 직접 구현하며 이해를 굳혀봅시다.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class SelfAttention(nn.Module):
    def __init__(self, embed_dim):
        super(SelfAttention, self).__init__()
        self.embed_dim = embed_dim

        # 입력 임베딩을 Q, K, V 벡터로 변환하기 위한 선형 레이어
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        # x: (배치 크기, 시퀀스 길이, 임베딩 차원)
        batch_size, seq_len, embed_dim = x.shape

        # 1. Q, K, V 벡터 생성
        Q = self.query(x)  # (B, N, D)
        K = self.key(x)    # (B, N, D)
        V = self.value(x)  # (B, N, D)

        # 2. 어텐션 스코어 계산 (Q * K^T) / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(1, 2)) / math.sqrt(self.embed_dim) # (B, N, N)

        # 3. 소프트맥스를 통해 어텐션 가중치 계산
        attention_weights = F.softmax(scores, dim=-1) # (B, N, N)

        # 4. Value 벡터에 가중치를 곱해 가중합 계산
        context_vector = torch.matmul(attention_weights, V) # (B, N, D)

        return context_vector, attention_weights

# 테스트
embed_dim = 64
seq_length = 10
batch_size = 1

# 더미 입력 데이터
x = torch.randn(batch_size, seq_length, embed_dim)

attention_module = SelfAttention(embed_dim)
context, weights = attention_module(x)

print("Input shape:", x.shape)
print("Output context vector shape:", context.shape)
print("Attention weights shape:", weights.shape)

Input shape: torch.Size([1, 10, 64])
Output context vector shape: torch.Size([1, 10, 64])
Attention weights shape: torch.Size([1, 10, 10])


### 3. 멀티헤드 어텐션(Multi-Head Attention): 여러 관점에서 바라보기

셀프 어텐션은 강력하지만, 한 번의 어텐션만으로는 문장의 다채로운 의미 관계를 모두 포착하기 어려울 수 있습니다. 

예를 들어 어떤 어텐션은 동사-목적어 관계에 집중하고, 다른 어텐션은 수식어-피수식어 관계에 집중하는 등 여러 관점에서 문장을 분석할 필요가 있습니다.

`멀티헤드 어텐션`은 이 아이디어를 구현한 것입니다. 

하나의 큰 Q, K, V를 사용하는 대신, 임베딩 차원을 여러 개의 '헤드(head)'로 나누어 각 헤드가 독립적으로 셀프 어텐션을 수행하게 합니다.

예를 들어 임베딩 차원이 512이고 헤드 수가 8개라면, 각 헤드는 64차원의 Q, K, V 벡터를 가지고 자신만의 어텐션을 계산합니다. 

이렇게 하면 8개의 서로 다른 '관점'에서 문장을 해석한 8개의 결과(문맥 벡터)가 나옵니다.

마지막으로, 이 8개의 결과를 다시 하나로 합치고(concatenate), 선형 레이어를 통과시켜 최종적인 하나의 문맥 벡터를 만들어냅니다. 

이는 여러 전문가의 의견을 종합하여 최종 결론을 내리는 것과 같습니다.

<img src="https://production-media.paperswithcode.com/methods/multi-head-attention_l1A3G7a.png" height="400">

#### 코드 실습: `MultiHeadAttention` 모듈 구현하기

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert embed_dim % num_heads == 0, "임베딩 차원은 헤드 수로 나누어 떨어져야 합니다."

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # 입력에서 모든 헤드에 대한 Q,K,V를 한번에 생성
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)

        # 헤드들의 출력을 합친 후, 최종 출력으로 변환하기 위한 레이어
        self.fc_out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        # x: (배치 크기, 시퀀스 길이, 임베딩 차원)
        batch_size, seq_len, _ = x.shape

        # 1. 전체 Q, K, V 생성
        Q = self.query(x) # (B, N, D)
        K = self.key(x)   # (B, N, D)
        V = self.value(x) # (B, N, D)

        # 2. 헤드 수에 맞게 벡터를 쪼갬 -> (B, num_heads, N, head_dim)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2) # transpose(1, 2) : 행과 열을 바꿈
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # 3. 각 헤드별로 어텐션 계산
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim) # transpose(-2, -1) : 행과 열을 바꿈
        attention_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attention_weights, V) # (B, h, N, head_dim)

        # 4. 쪼개진 헤드들을 다시 하나로 합침
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.embed_dim) # contiguous() : 메모리 연속성 보장

        # 5. 최종 선형 레이어 통과
        output = self.fc_out(context) # (B, N, D)

        return output, attention_weights

# 테스트
embed_dim = 256
num_heads = 8

x = torch.randn(batch_size, seq_length, embed_dim)

multi_head_attention = MultiHeadAttention(embed_dim, num_heads)
output, weights = multi_head_attention(x)

print("Input shape:", x.shape)
print("Output context vector shape:", output.shape)
print("Attention weights shape:", weights.shape)
# (배치 크기, 헤드 수, 시퀀스 길이, 시퀀스 길이)

Input shape: torch.Size([1, 10, 256])
Output context vector shape: torch.Size([1, 10, 256])
Attention weights shape: torch.Size([1, 8, 10, 10])


### 4. 포지셔널 인코딩(Positional Encoding): 순서 감각을 불어넣다

트랜스포머의 어텐션 메커니즘은 모든 단어를 동시에 처리하기 때문에, "고양이가 쥐를 쫓는다"와 "쥐가 고양이를 쫓는다"를 구분하지 못합니다. 

즉, 단어의 `위치 정보`가 없습니다.

`포지셔널 인코딩`은 이 문제를 해결하기 위해 각 단어의 임베딩에 그 단어의 위치를 나타내는 고유한 벡터를 더해주는 기법입니다. 

이렇게 하면 모델은 단어의 의미뿐만 아니라 문장 내에서의 위치 정보까지 함께 학습할 수 있게 됩니다.

<img src="https://i.sstatic.net/E1aEA.jpg" width="600">

원 논문에서는 sin, cos 함수를 이용한 고정된 값을 사용했지만, 위치 정보 자체를 학습 가능한 파라미터로 만드는 `학습 가능한(learnable) 포지셔널 임베딩` 방식도 널리 쓰입니다. 

우리는 구현의 편의를 위해 후자의 방식을 채택하겠습니다. 

`nn.Embedding` 레이어를 하나 더 만들어, 0번 위치, 1번 위치, 2번 위치... 에 해당하는 벡터를 모델이 직접 학습하게 만드는 것입니다.

In [5]:
import copy

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term) # 짝수 인덱스
        pe[:, 1::2] = torch.cos(position * div_term) # 홀수 인덱스
        
        pe = pe.unsqueeze(0) # (1, max_len, d_model)
        self.register_buffer('pe', pe) # 학습되지 않는 파라미터로 등록

    def forward(self, x):
        # x: (Batch, SeqLen, d_model)
        # 입력 x의 시퀀스 길이에 맞춰 PE를 더해줌
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)
    

In [7]:
# 포지셔널 인코딩 테스트
d_model = 512
max_len = 100
pos_encoding = PositionalEncoding(d_model, max_len=max_len)

# 테스트 입력 생성
batch_size = 2
seq_len = 10
test_input = torch.randn(batch_size, seq_len, d_model)

# 포지셔널 인코딩 적용
output = pos_encoding(test_input)

print(f"입력 형태: {test_input.shape}")
print(f"출력 형태: {output.shape}")
print(f"포지셔널 인코딩이 적용되었는지 확인:")
print(f"입력과 출력이 다른가? {not torch.allclose(test_input, output)}")

입력 형태: torch.Size([2, 10, 512])
출력 형태: torch.Size([2, 10, 512])
포지셔널 인코딩이 적용되었는지 확인:
입력과 출력이 다른가? True


#### 4-1. Position-wise Feed-Forward Networks

어텐션을 통과한 각 단어 벡터를 개별적으로 더 깊게 처리해주는 간단한 2층 신경망입니다. 모든 위치의 단어들이 동일한 가중치(FFN)를 공유합니다.

In [ ]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.w_2(self.dropout(F.relu(self.w_1(x))))


### 5. 인코더 블록 조립하기: 완성된 부품의 결합

이제 우리는 트랜스포머의 핵심 부품을 모두 손에 넣었습니다. 

트랜스포머의 `인코더 블록`은 이 부품들을 다음과 같은 순서로 조립하여 만듭니다.

1.  `멀티헤드 셀프 어텐션 (Multi-Head Self-Attention)`

2.  `Add & Norm (잔차 연결 & 층 정규화)`: 어텐션을 통과한 출력과 어텐션에 들어가기 전의 원래 입력을 더해주고(잔차 연결), 그 결과를 정규화합니다. 이는 깊은 신경망의 학습을 안정시키고 성능을 높이는 매우 중요한 기법입니다.
3.  `피드포워드 신경망 (Feed-Forward Neural Network)`: 문맥이 반영된 각 단어 벡터를 한 번 더 심층적으로 처리해주는 간단한 2층 신경망입니다.
4.  `Add & Norm`: 피드포워드 신경망을 통과한 출력과 그 입력값을 다시 더해주고(잔차 연결), 정규화합니다.
   

<img src="https://miro.medium.com/v2/resize:fit:1400/1*8lAtTYcAgw5nBII5kdpFBg.png" height="600">

이 `[어텐션 -> Add&Norm -> 피드포워드 -> Add&Norm]` 구조가 바로 트랜스포머 인코더 1개 층입니다. 

실제 트랜스포머 모델은 이런 인코더 층을 여러 개(보통 6개 이상) 쌓아서 만듭니다.

#### 코드 실습: `TransformerEncoderLayer` 구현하기

In [3]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1):
        super(TransformerEncoderLayer, self).__init__()
        self.attention = MultiHeadAttention(embed_dim, num_heads)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        # 1. 멀티헤드 어텐션
        attn_output, _ = self.attention(x)

        # 2. Add & Norm (잔차 연결 + 층 정규화)
        # 어텐션의 입력(x)과 출력(attn_output)을 더함
        x = self.norm1(x + self.dropout(attn_output))

        # 3. 피드포워드 신경망
        ff_output = self.feed_forward(x)

        # 4. Add & Norm
        x = self.norm2(x + self.dropout(ff_output))

        return x

# 테스트
embed_dim = 256
num_heads = 8
ff_dim = 1024 # 피드포워드 신경망의 내부 차원

encoder_layer = TransformerEncoderLayer(embed_dim, num_heads, ff_dim)
x = torch.randn(batch_size, seq_length, embed_dim)
output = encoder_layer(x)

print("Input shape:", x.shape)
print("Encoder layer output shape:", output.shape)

Input shape: torch.Size([1, 10, 256])
Encoder layer output shape: torch.Size([1, 10, 256])
